# P01 (basic) — point cloud basics: neighbourhoods, downsampling & normals

**Module 20 — 3D Point Cloud Processing**

Before you can register or segment point clouds you need three basic operations:
1. Find **neighbourhoods** efficiently (kNN / radius) — via a **kd-tree**.
2. **Downsampling** (voxel grid) — fewer points, uniform density.
3. **Estimate normals** — via a **local PCA** (eigendecomposition of the covariance matrix).

You build all three. The neat part for the normals: we sample a **sphere** whose true normals we
know (they point radially outwards) — so you can **check your estimate against the ground truth**.

### Goal
- use kd-tree neighbourhoods with `scipy.spatial.cKDTree`,
- implement **voxel downsampling**,
- estimate **normals + curvature** through the local covariance PCA (ch. 5 of the script),
- validate the estimate against the analytic sphere normals.

### Format
Jupyter notebook — point cloud work thrives on 3D visualisation next to the computation.

### Prior knowledge
PCA / eigendecomposition (module 05), ch. 3-5 of the module 20 script.

### Tasks
Most cells are given; at the `# TODO` spots you implement downsampling and normal estimation. The solution is in `solution/`.

## Setup
`numpy`, `scipy` (kd-tree), `matplotlib` (3D) — all in the `.venv`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
from scipy.spatial import cKDTree
np.set_printoptions(precision=3, suppress=True)
rng = np.random.default_rng(0)

## Part A — a synthetic point cloud (sphere with noise)

We sample points uniformly on a sphere (centre `c`, radius `R`) and add radial Gaussian noise.
The **true normal** at a point is the radial direction
$(\mathbf p - \mathbf c)/\lVert\mathbf p - \mathbf c\rVert$ — we keep it as the ground truth.
(Cell given.)

In [ ]:
def sample_sphere(n, center, R, noise=0.01, rng=rng):
    # uniformly distributed directions: normalised Gaussian vectors
    dirs = rng.normal(size=(n, 3))
    dirs /= np.linalg.norm(dirs, axis=1, keepdims=True)
    radii = R + rng.normal(0, noise, n)          # radial noise
    pts = center + dirs * radii[:, None]
    gt_normals = dirs                            # true (outward) normal = direction
    return pts, gt_normals

center = np.array([0.0, 0.0, 0.0]); R = 1.0
pts, gt_normals = sample_sphere(2500, center, R, noise=0.01)
print("point cloud:", pts.shape)

fig = plt.figure(figsize=(6, 6)); ax = fig.add_subplot(111, projection="3d")
ax.scatter(pts[:, 0], pts[:, 1], pts[:, 2], s=3, alpha=0.4)
ax.set_title("sphere point cloud (2500 points)")
try: ax.set_box_aspect((1, 1, 1))
except Exception: pass
plt.show()

## Part B — neighbourhoods with the kd-tree

The **kd-tree** answers kNN and radius queries in $O(\log n)$ on average instead of $O(n)$.
`scipy.spatial.cKDTree` builds it in $O(n\log n)$. (Cell given — watch the outputs.)

In [ ]:
tree = cKDTree(pts)

# kNN: the 8 nearest neighbours of the first point
dist, idx = tree.query(pts[0], k=8)
print("8-NN of point 0: indices", idx)
print("distances       ", dist.round(3))

# radius search: all points within radius 0.1 of point 0
nb = tree.query_ball_point(pts[0], r=0.1)
print(f"points within radius 0.1 of point 0: {len(nb)}")

## Part C — voxel downsampling

Lay a grid of edge length `voxel` over the cloud and **replace all points inside a voxel by their
centroid**. Result: at most one point per occupied voxel, uniform density.

**Your task:** implement `voxel_downsample`. Steps:
- voxel index of each point: `np.floor((pts - pts.min(0)) / voxel)` as an integer triple.
- group points with the same voxel index, take the **mean** of each group.
(Hint: `np.unique(..., axis=0, return_inverse=True)` gives group labels.)

In [ ]:
def voxel_downsample(pts, voxel):
    keys = np.floor((pts - pts.min(0)) / voxel).astype(np.int64)   # (n,3) voxel indices
    # TODO: group by the unique voxel key and average the points within each group
    # uniq, inv = np.unique(keys, axis=0, return_inverse=True)
    # sum per group / count per group
    ...  # TODO
    return down  # (m,3) with m <= n

down = voxel_downsample(pts, voxel=0.15)
print(f"before {len(pts)} points -> after {len(down)} points (voxel 0.15)")

**Expectation.** Of the 2500 points a few hundred remain depending on the voxel size (at 0.15
roughly 680). The sphere shape is preserved, the density becomes more uniform.

## Part D — normal estimation via a local PCA

The core (script ch. 5). For each point:
1. fetch the $k$ nearest neighbours (kd-tree),
2. their **covariance matrix** $\mathbf C = \frac{1}{k}\sum (\mathbf q-\bar{\mathbf q})(\mathbf q-\bar{\mathbf q})^\top$,
3. eigendecomposition; the **normal = eigenvector of the smallest eigenvalue** (direction of minimal variance),
4. **curvature** $\sigma = \lambda_0/(\lambda_0+\lambda_1+\lambda_2)$.

We orient the sign **outwards** (away from the centre): if $\mathbf n\cdot(\mathbf p-\mathbf c)<0$,
flip $\mathbf n$.

**Your task:** fill in the covariance, the normal and the curvature.

In [ ]:
def estimate_normals(pts, k=16, viewpoint_outward_center=None):
    tree = cKDTree(pts)
    normals = np.zeros_like(pts)
    curvature = np.zeros(len(pts))
    for i, p in enumerate(pts):
        _, idx = tree.query(p, k=k)
        nb = pts[idx]
        nb_c = nb - nb.mean(0)                     # centre
        # TODO: covariance matrix C (3x3)
        C = ...  # TODO
        evals, evecs = np.linalg.eigh(C)           # eigh: ascending eigenvalues
        # TODO: normal = eigenvector of the SMALLEST eigenvalue (column 0)
        n = ...  # TODO
        # TODO: curvature = smallest eigenvalue / sum of the eigenvalues
        curv = ...  # TODO
        # orient outwards (away from the centre), if a centre is given
        if viewpoint_outward_center is not None:
            if np.dot(n, p - viewpoint_outward_center) < 0:
                n = -n
        normals[i] = n; curvature[i] = curv
    return normals, curvature

est_normals, curv = estimate_normals(pts, k=16, viewpoint_outward_center=center)

# validation against the true sphere normals: angular error
cos = np.clip(np.sum(est_normals * gt_normals, axis=1), -1, 1)
ang_err_deg = np.rad2deg(np.arccos(cos))
print(f"normal angular error: median {np.median(ang_err_deg):.2f} deg, "
      f"mean {ang_err_deg.mean():.2f} deg")
print(f"curvature (sphere ~ constant, small): mean {curv.mean():.4f}")

**Expectation / self-check.** The **median angular error** of the estimated normals should be
small (roughly **1-5 degrees** at k=16 and this noise level) — so your local PCA reconstructs the
sphere normals well. The **curvature** is small and roughly constant across the sphere (a smooth
surface); at an edge/corner it would be large. Increase the noise or shrink `k` and the error grows
— normal estimation is a bias-variance trade-off in the neighbourhood size.

## Part E — visualise the normals

(Given.) We draw a subset of the points with their estimated normals as arrows — they should all
point nicely radially outwards.

In [ ]:
sel = rng.choice(len(pts), 200, replace=False)
fig = plt.figure(figsize=(7, 7)); ax = fig.add_subplot(111, projection="3d")
ax.scatter(pts[:, 0], pts[:, 1], pts[:, 2], s=2, alpha=0.15, color="gray")
ax.quiver(pts[sel, 0], pts[sel, 1], pts[sel, 2],
          est_normals[sel, 0], est_normals[sel, 1], est_normals[sel, 2],
          length=0.25, color="crimson", linewidth=0.7)
ax.set_title("estimated normals (pointing radially outwards)")
try: ax.set_box_aspect((1, 1, 1))
except Exception: pass
plt.show()

## Conclusion

You have built the three basic operations of every point cloud pipeline:
- **kd-tree** neighbourhoods (the engine underneath everything that follows),
- **voxel downsampling** (preprocessing),
- **normals + curvature via a local PCA** — validated against the sphere ground truth.

These building blocks are the precondition for the next projects: **ICP registration** (medium)
needs neighbourhoods for the correspondence search, and the **segmentation pipeline** (final) needs
normals and neighbourhoods for RANSAC + clustering.